In [1]:
from urllib.request import urlopen


URL = 'https://dev.w3.org/html5/html-author/charref'
with urlopen(URL) as r:
    print(f'[{r.status}] {r.url}')
    HTML = r.read()

print(f'{len(HTML)}B HTML fetched')

[200] https://dev.w3.org/html5/html-author/charref
500260B HTML fetched


In [2]:
from collections import namedtuple
import re
import sys
from bs4 import BeautifulSoup as BS


Entity = namedtuple('Entity', 'name hex dec char entity')


soup = BS(HTML)
entities = []
for row in soup.find_all('tr'):
    m = re.match(r'U\+([0-9A-Z]+) (\w+)', row.get('title', ''))
    if not m:
        print(f'no title in row: {repr(row)}', file=sys.stderr)
        continue

    cp, name = m.groups()
    dec = int(cp, 16)
    fields = [name, cp, dec]

    cells = row.find_all('td')
    if len(cells) != 5:
        print(f'{len(cells)} in row, not 5: {repr(row)}')
        continue
    
    # character literal
    fields.append(cells[0].get_text())
    # named entity
    m = re.match(r'&(.+?);', cells[1].get_text())
    if not m:
        print(f'no named entity found: {repr(row)}')
        continue
    fields.append(m.group(1))

    entities.append(Entity(*fields))

entities.sort(key=lambda e: e.dec)
print(f'{len(entities)} entities')

1448 entities


In [4]:
import json

go = [
    '// !!!this file was auto-generated!!!', '',
    'package main', '',
    '// NamedHTMLEntities maps Unicode codepoints to HTML entity names.',
    'var NamedHTMLEntities = map[string]string{',
]

for i, e in enumerate(entities):
    go.append(f'\t"{e.hex}": "{e.entity}",')

go.append('}')

with open('html_entities_gen.go', 'w') as fp:
    fp.write('\n'.join(go))


d = {e.hex:e.entity for e in entities}
with open('html_entities.json', 'w') as fp:
    json.dump(d, fp, indent=2, sort_keys=True, separators=(',', ': '))
